# Capítulo 2 — Modelos Clássicos de Machine Learning para Defesa

**Inteligência Artificial e Machine Learning Avançado para Defesa** — Prof. Cristiano Alves · Quantum Strategic AI

---

### 🎯 Objetivos do capítulo

Ao final deste notebook, você será capaz de:

1. **Formular e treinar** modelos de **regressão linear** e **regressão logística**, distinguindo com clareza quando o problema é de *previsão de quantidade* e quando é de *decisão entre classes*;
2. **Explicar** o funcionamento de **árvores de decisão** e **florestas aleatórias** (*random forests*), extraindo delas medidas de **importância de variáveis** úteis à auditoria;
3. **Compreender** a formulação das **máquinas de vetores de suporte (SVM)**, o conceito de **margem máxima** e o papel dos **kernels** na separação de dados não lineares;
4. **Escolher métricas** adequadas a problemas de classificação com **classes desbalanceadas** — situação típica na detecção de ameaças — e interpretar a **matriz de confusão** em termos de custo operacional;
5. **Construir**, em `scikit-learn`, um **pipeline comparativo** que confronte vários modelos clássicos sobre um mesmo problema de classificação a partir de dados de sensores.

> No Capítulo 1, organizamos o vocabulário e fixamos o esqueleto do trabalho: as três tradições da IA, os três regimes de aprendizado e o pipeline de cinco etapas. Neste capítulo, **descemos do panorama à prática**. Os modelos clássicos são, na maioria dos problemas reais de defesa, a **baseline honesta** contra a qual qualquer arquitetura profunda terá de se justificar — e, com frequência, a escolha final, por serem mais **auditáveis**, mais **robustos com poucos dados** e mais **fáceis de manter em produção**.
>
> Em sistemas críticos, a pergunta certa não é *"qual o modelo mais sofisticado que consigo treinar?"*, mas **"qual o modelo mais simples que resolve o problema com garantias suficientes?"**. Este capítulo é, em larga medida, uma defesa dessa pergunta.

## Preparação do ambiente

O Google Colab já traz instaladas todas as bibliotecas necessárias (`scikit-learn`, `numpy`, `matplotlib`). A célula abaixo fixa as versões usadas e a **semente aleatória** — requisito de reprodutibilidade que, em sistemas de defesa, não é opcional: *dados os mesmos dados, os mesmos resultados*.

In [ ]:
# Preparacao do ambiente: importacoes e semente de reprodutibilidade
import numpy as np
import matplotlib.pyplot as plt
import sklearn

SEMENTE = 0
np.random.seed(SEMENTE)

print(f"numpy        {np.__version__}")
print(f"scikit-learn {sklearn.__version__}")
print("Ambiente pronto.")

---
## 2.1 Regressão linear: a baseline de toda previsão

A regressão linear é o modelo **mais simples** do aprendizado supervisionado — e, não por acaso, o **ponto de partida correto** para qualquer problema de *regressão* (previsão de uma quantidade contínua). Sua simplicidade não é defeito: **é uma virtude**. Um modelo que se entende completamente, cujos coeficientes têm interpretação direta e cujo comportamento se prevê fora dos dados de treino é, em muitos contextos operacionais, preferível a uma alternativa opaca de desempenho marginalmente superior.

### 2.1.1 Formulação

Dado um vetor de entrada $\mathbf{x} = (x_1, x_2, \ldots, x_d)$ com $d$ atributos, a regressão linear modela a saída $\hat{y}$ como uma **combinação linear** das entradas:

$$\hat{y} = w_1 x_1 + w_2 x_2 + \cdots + w_d x_d + b = \mathbf{w}^\top \mathbf{x} + b$$

em que $\mathbf{w}$ é o vetor de **pesos** (coeficientes) e $b$ é o **viés** (*intercept*). Treinar o modelo significa encontrar $\mathbf{w}$ e $b$ que minimizam o **erro quadrático médio** (MSE) sobre os $n$ exemplos de treinamento:

$$\mathrm{MSE}(\mathbf{w}, b) = \frac{1}{n} \sum_{i=1}^{n} \left( y_i - \mathbf{w}^\top \mathbf{x}_i - b \right)^2$$

Esse problema tem **solução fechada** (as *equações normais*), o que torna a regressão linear excepcionalmente **barata de treinar** e **perfeitamente reproduzível**.

### 2.1.2 Interpretação dos coeficientes

A grande força operacional da regressão linear é a **leitura direta de seus coeficientes**: cada $w_j$ exprime *quanto a saída prevista varia para um aumento unitário no atributo* $x_j$, mantidos os demais constantes.

> **🛡️ Contexto de defesa** — Em **manutenção preditiva** de plataformas militares (viaturas, aeronaves, motores navais), a regressão linear é frequentemente o primeiro modelo a se testar para estimar grandezas como *horas até a próxima intervenção* ou *consumo esperado em uma missão*. Sua transparência é decisiva: quando o sistema de apoio à decisão logística recomenda antecipar a manutenção de um meio, o oficial responsável precisa poder **auditar a recomendação** — e um modelo cujos coeficientes se leem em linguagem de engenharia permite exatamente isso. Uma rede neural com desempenho 2% superior, mas inauditável, raramente compensa a perda de rastreabilidade.

### 2.1.3 Em código

A **Listagem 2.1** ajusta uma regressão linear a um conjunto sintético que simula a relação entre o **perfil de uma missão** (distância, carga, velocidade média) e o **consumo de combustível** observado. O exemplo introduz o padrão que se repetirá em todo o capítulo: *separar treino e teste, ajustar o modelo, prever e medir o erro*.

In [ ]:
# Listagem 2.1 - Regressao linear para estimativa de consumo.
# Estimativa de consumo de combustivel a partir do perfil de missao.
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

rng = np.random.default_rng(0)
n = 400
distancia  = rng.uniform(50, 800, n)   # km
carga      = rng.uniform(0.5, 8.0, n)  # toneladas
velocidade = rng.uniform(40, 90, n)    # km/h

# Relacao "verdadeira" + ruido (desconhecida do modelo)
consumo = (0.18 * distancia + 7.5 * carga + 0.9 * velocidade
           + rng.normal(0, 12, n))     # litros

X = np.column_stack([distancia, carga, velocidade])
y = consumo

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0)

modelo = LinearRegression()
modelo.fit(X_tr, y_tr)
y_pred = modelo.predict(X_te)

print("Coeficientes:", np.round(modelo.coef_, 3))
print("Intercepto:  ", round(modelo.intercept_, 2))
print("MAE (litros):", round(mean_absolute_error(y_te, y_pred), 2))
print("R2:          ", round(r2_score(y_te, y_pred), 3))

**Observe:** os coeficientes estimados aproximam-se dos valores que **geraram** os dados (`0.18`, `7.5`, `0.9`) — o modelo *recuperou a física do problema* a partir de exemplos ruidosos —, e o coeficiente de determinação $R^2$ situa-se próximo de 0,9. Mais relevante que os números é o **método**: a regressão linear oferece, de graça, uma baseline interpretável que qualquer modelo mais complexo terá de superar de forma convincente.

A célula seguinte visualiza a qualidade do ajuste: à esquerda, *previsto vs. real* (quanto mais próximo da diagonal, melhor); à direita, os **resíduos** (erros), que devem se distribuir em torno de zero, **sem estrutura** — resíduo com padrão é sintoma de que o modelo deixou informação na mesa.

In [ ]:
# Visualizacao do ajuste: previsto vs. real e residuos
fig, eixos = plt.subplots(1, 2, figsize=(11, 4))

eixos[0].scatter(y_te, y_pred, alpha=0.6, edgecolor="k", linewidth=0.3)
limites = [min(y_te.min(), y_pred.min()), max(y_te.max(), y_pred.max())]
eixos[0].plot(limites, limites, "r--", lw=1.5, label="previsao perfeita")
eixos[0].set_xlabel("Consumo real (litros)")
eixos[0].set_ylabel("Consumo previsto (litros)")
eixos[0].set_title("Previsto vs. real (teste)")
eixos[0].legend()

residuos = y_te - y_pred
eixos[1].scatter(y_pred, residuos, alpha=0.6, edgecolor="k", linewidth=0.3)
eixos[1].axhline(0, color="r", ls="--", lw=1.5)
eixos[1].set_xlabel("Consumo previsto (litros)")
eixos[1].set_ylabel("Residuo (real - previsto)")
eixos[1].set_title("Residuos: devem oscilar em torno de zero, sem padrao")

plt.tight_layout()
plt.show()

> **✅ Boa prática** — **Nunca apresente um modelo complexo sem compará-lo a uma baseline simples.** Para regressão, a baseline mínima é *prever sempre a média do alvo*; o passo seguinte é uma regressão linear. Se um modelo sofisticado não supera de modo claro essas referências, ou o problema é intrinsecamente difícil, ou — bem mais comum — **há erro no pipeline**. A baseline é, ao mesmo tempo, termo de comparação e instrumento de depuração.

A célula abaixo materializa essa hierarquia de baselines com o `DummyRegressor`:

In [ ]:
# Hierarquia de baselines: media do alvo -> regressao linear
from sklearn.dummy import DummyRegressor

baseline_media = DummyRegressor(strategy="mean").fit(X_tr, y_tr)

print("MAE (litros) no conjunto de teste:")
print(f"  Baseline 'sempre a media': {mean_absolute_error(y_te, baseline_media.predict(X_te)):7.2f}")
print(f"  Regressao linear:          {mean_absolute_error(y_te, y_pred):7.2f}")
print()
print("Se um modelo sofisticado nao superar CLARAMENTE esses numeros,")
print("desconfie do modelo -- ou do pipeline.")

---
## 2.2 Regressão logística: da quantidade à decisão

A maior parte dos problemas operacionais de defesa não pede a previsão de uma quantidade, mas uma **decisão entre classes**: *este contato é hostil ou neutro? esta imagem contém um alvo? este tráfego de rede é legítimo ou anômalo?* Para esses problemas de **classificação**, o modelo clássico fundamental é a **regressão logística** — que, a despeito do nome, *é um classificador*.

### 2.2.1 Da combinação linear à probabilidade

A ideia é elegante: parte-se da mesma combinação linear da regressão, $z = \mathbf{w}^\top \mathbf{x} + b$, mas passa-se o resultado por uma função que o comprime ao intervalo $[0, 1]$, interpretável como **probabilidade** — a **sigmoide** (ou logística):

$$\sigma(z) = \frac{1}{1 + e^{-z}}, \qquad P(y = 1 \mid \mathbf{x}) = \sigma\left(\mathbf{w}^\top \mathbf{x} + b\right)$$

A decisão final se obtém por um **limiar** (*threshold*): tipicamente, classifica-se como positivo se $P(y=1 \mid \mathbf{x}) \geq 0{,}5$. **Esse limiar não é sagrado** — ajustá-lo é uma das alavancas operacionais mais importantes, como veremos.

O treinamento minimiza a **entropia cruzada** (*log-loss*), que penaliza com severidade as previsões *confiantes e erradas*:

$$\mathcal{L}(\mathbf{w}, b) = -\frac{1}{n} \sum_{i=1}^{n} \Big[ y_i \log \hat{p}_i + (1 - y_i) \log(1 - \hat{p}_i) \Big]$$

In [ ]:
# A funcao sigmoide: comprime qualquer valor real ao intervalo [0, 1]
z = np.linspace(-8, 8, 300)
sigma = 1 / (1 + np.exp(-z))

plt.figure(figsize=(8, 4))
plt.plot(z, sigma, lw=2.5)
plt.axhline(0.5, color="gray", ls=":", lw=1)
plt.axvline(0.0, color="gray", ls=":", lw=1)
plt.annotate("limiar padrao: sigma(z) = 0.5 quando z = 0",
             xy=(0, 0.5), xytext=(1.5, 0.30),
             arrowprops=dict(arrowstyle="->", color="gray"))
plt.fill_between(z, sigma, 0.5, where=(sigma >= 0.5), alpha=0.15, color="tab:red",
                 label="regiao classificada como POSITIVO (>= 0.5)")
plt.xlabel("z = w'x + b  (combinacao linear)")
plt.ylabel("sigma(z) = P(y = 1 | x)")
plt.title("Sigmoide: da combinacao linear a probabilidade")
plt.legend(loc="upper left")
plt.tight_layout()
plt.show()

### 2.2.2 A probabilidade como informação operacional

A regressão logística não devolve apenas um rótulo: devolve uma **probabilidade** (aproximadamente calibrada). Em um sistema de apoio à decisão, saber que um contato tem **0,55** de probabilidade de ser hostil é operacionalmente distinto de saber que tem **0,98** — ainda que ambos cruzem o limiar de 0,5. **A probabilidade é, ela própria, informação para o operador.**

> **📌 Nota** — Modelos baseados em combinação linear — regressão linear, regressão logística, SVM — são **sensíveis à escala dos atributos**. Um atributo em metros e outro em quilômetros entram na mesma soma ponderada com pesos de ordens de grandeza diferentes, o que prejudica a otimização. Por isso, **padronizar os atributos** (subtrair a média, dividir pelo desvio padrão) antes do treino é prática indispensável. Modelos baseados em **árvores**, que veremos a seguir, são imunes a esse problema — uma de suas vantagens práticas.

### 2.2.3 Classes desbalanceadas e o custo do erro

Aqui surge a **primeira armadilha séria** da classificação operacional. Em defesa, as classes de interesse são, com frequência, **raras**: a vasta maioria dos contatos é neutra, a vasta maioria do tráfego é legítima, a vasta maioria das imagens não contém alvo. Um classificador que simplesmente respondesse *"classe majoritária"* a tudo atingiria acurácia altíssima — e seria **operacionalmente inútil**, pois nunca detectaria o que importa.

A **Listagem 2.2** treina uma regressão logística sobre um conjunto desbalanceado que simula a **triagem de contatos**, com apenas **12% de exemplos hostis**, e exibe a **matriz de confusão** e o **relatório por classe** — não a acurácia agregada.

In [ ]:
# Listagem 2.2 - Regressao logistica em problema desbalanceado de triagem.
# Triagem de contatos com classes desbalanceadas (12% hostis).
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report

X, y = make_classification(
    n_samples=2000, n_features=10, n_informative=5,
    weights=[0.88, 0.12], random_state=0   # 12% da classe positiva (hostil)
)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y
)

escalador = StandardScaler().fit(X_tr)
X_tr_n, X_te_n = escalador.transform(X_tr), escalador.transform(X_te)

# class_weight="balanced" compensa o desbalanceamento das classes
modelo = LogisticRegression(max_iter=1000, class_weight="balanced")
modelo.fit(X_tr_n, y_tr)
y_pred = modelo.predict(X_te_n)

print("Matriz de confusao:\n", confusion_matrix(y_te, y_pred))
print(classification_report(y_te, y_pred, digits=3,
      target_names=["neutro", "hostil"]))

**Observe:** o parâmetro `class_weight="balanced"` instrui o modelo a **ponderar mais os erros sobre a classe rara**, deslocando o equilíbrio em favor da **revocação** (*recall*) da classe hostil — isto é, da fração de ameaças efetivamente detectadas. Esse deslocamento **aumenta os falsos positivos** (alarmes sobre contatos neutros), e essa é exatamente a troca que o projetista precisa calibrar conscientemente, em diálogo com o operador.

A célula abaixo torna a matriz de confusão **legível em termos operacionais** — cada quadrante tem um nome e um custo:

In [ ]:
# Matriz de confusao com leitura operacional de cada quadrante
from sklearn.metrics import ConfusionMatrixDisplay

fig, eixo = plt.subplots(figsize=(5.5, 5))
ConfusionMatrixDisplay.from_predictions(
    y_te, y_pred, display_labels=["neutro", "hostil"],
    cmap="Blues", colorbar=False, ax=eixo
)
eixo.set_title("Matriz de confusao da triagem de contatos")
eixo.set_xlabel("Classe prevista")
eixo.set_ylabel("Classe real")
plt.tight_layout()
plt.show()

vn, fp, fn, vp = confusion_matrix(y_te, y_pred).ravel()
print(f"Verdadeiros negativos (neutro OK):        {vn:4d}")
print(f"Falsos positivos (ALARME FALSO):          {fp:4d}  -> sobrecarga do operador")
print(f"Falsos negativos (AMEACA NAO DETECTADA):  {fn:4d}  -> risco operacional grave")
print(f"Verdadeiros positivos (ameaca detectada): {vp:4d}")

> **⚠️ Armadilha comum** — **A acurácia agregada é a métrica mais enganosa em problemas de classe rara.** Com 12% de exemplos hostis, um classificador que *nunca dispara o alarme* acerta 88% das vezes — e é inútil. Nunca avalie um problema desbalanceado por acurácia. Use **precisão** e **revocação** por classe, o **F1-score**, a **matriz de confusão** e, quando o limiar for ajustável, a **curva precision–recall**. Em defesa, o ponto é ainda mais agudo: **falsos negativos** (ameaças não detectadas) e **falsos positivos** (alarmes falsos) têm custos operacionais radicalmente diferentes — e é esse *custo*, não a acurácia, que deve guiar a escolha do limiar.

A célula seguinte demonstra a armadilha na prática e, em seguida, usa o **limiar de decisão como alavanca operacional**:

In [ ]:
# Parte 1 - A armadilha: o "classificador" que nunca dispara o alarme
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score

mudo = DummyClassifier(strategy="most_frequent").fit(X_tr_n, y_tr)
print("Classificador que SEMPRE responde 'neutro':")
print(f"  Acuracia:            {accuracy_score(y_te, mudo.predict(X_te_n)):.3f}  <- parece otimo...")
print(f"  Revocacao (hostil):  {recall_score(y_te, mudo.predict(X_te_n)):.3f}  <- ...e inutil.")
print()

# Parte 2 - O limiar como alavanca: variando o ponto de corte da probabilidade
probas = modelo.predict_proba(X_te_n)[:, 1]   # P(hostil | x)

print("Limiar | Precisao (hostil) | Revocacao (hostil) | Alarmes emitidos")
for limiar in [0.30, 0.50, 0.70]:
    y_lim = (probas >= limiar).astype(int)
    p = precision_score(y_te, y_lim)
    r = recall_score(y_te, y_lim)
    print(f"  {limiar:.2f} |       {p:.3f}       |       {r:.3f}        |      {y_lim.sum():4d}")

print()
print("Baixar o limiar aumenta a revocacao (menos ameacas escapam)")
print("ao preco de mais alarmes falsos. O limiar certo depende do CUSTO")
print("operacional de cada erro -- nao existe resposta universal.")

> **✏️ Experimente:** na célula acima, acrescente os limiares `0.10` e `0.90` à lista. Em que limiar a revocação chega perto de 1,0? Quantos alarmes o operador teria de triar nesse caso? Essa é, em miniatura, a negociação que projetista e operador fazem em todo sistema real de triagem.

---
## 2.3 Árvores de decisão e florestas aleatórias

Os modelos lineares supõem que a relação entre atributos e saída é, em alguma escala, **linear**. Quando essa hipótese falha — quando há **interações complexas** entre atributos, **limiares abruptos**, regiões de decisão irregulares —, uma família diferente se mostra mais adequada: os **modelos baseados em árvores**.

### 2.3.1 A árvore de decisão

Uma árvore de decisão classifica uma entrada por uma **sequência de perguntas binárias** sobre seus atributos: *a velocidade é maior que 40 nós?*; em caso afirmativo, *a assinatura acústica está na banda X?*; e assim por diante, até uma **folha** que atribui a classe. Cada nó interno escolhe o atributo e o limiar que **melhor separam as classes**, segundo uma medida de **impureza** — a mais comum é o **índice de Gini**:

$$G = 1 - \sum_{k=1}^{K} p_k^2$$

em que $p_k$ é a proporção de exemplos da classe $k$ no nó. Gini é **zero quando o nó é puro** (uma só classe) e máximo quando as classes se distribuem por igual.

A virtude das árvores é dupla: capturam **relações não lineares e interações** sem que se precise especificá-las, e produzem um modelo **legível** — o caminho da raiz à folha é uma regra explícita, auditável, próxima da forma como um analista descreveria o próprio raciocínio. Sua fraqueza é a tendência ao **sobreajuste** (*overfitting*): uma árvore profunda o bastante *memoriza o ruído* do treino, decorando os dados em vez de aprender o padrão.

A célula abaixo treina uma árvore rasa e a **desenha** — repare que cada nó é uma pergunta legível:

In [ ]:
# Uma arvore de decisao rasa e legivel (max_depth=3)
from sklearn.datasets import make_classification
from sklearn.tree import DecisionTreeClassifier, plot_tree

X_arv, y_arv = make_classification(
    n_samples=600, n_features=4, n_informative=3, n_redundant=0,
    random_state=SEMENTE
)
nomes_atributos = ["velocidade", "banda_acustica", "rcs_radar", "altitude"]

arvore = DecisionTreeClassifier(max_depth=3, random_state=SEMENTE)
arvore.fit(X_arv, y_arv)

plt.figure(figsize=(14, 6))
plot_tree(arvore, feature_names=nomes_atributos,
          class_names=["neutro", "hostil"], filled=True,
          rounded=True, fontsize=9, impurity=True)
plt.title("Arvore de decisao: cada caminho raiz->folha e uma regra auditavel")
plt.show()

print("Leitura de exemplo: 'SE velocidade <= a E banda_acustica > b ENTAO hostil'")
print("-- proxima da forma como um analista descreveria o proprio raciocinio.")

### 2.3.2 Da árvore à floresta

A **floresta aleatória** (*random forest*) corrige a fragilidade da árvore isolada por uma ideia simples e poderosa: em vez de uma árvore, treinam-se **muitas** — centenas —, cada uma sobre uma **amostra aleatória dos dados** (com reposição, técnica chamada *bagging*) e considerando, em cada divisão, apenas um **subconjunto aleatório dos atributos**. A predição final é o **voto majoritário** das árvores. O efeito é estatístico: erros individuais, **descorrelacionados** entre árvores, tendem a **se cancelar no agregado**, e o conjunto generaliza muito melhor que qualquer árvore isolada.

> **🛡️ Contexto de defesa** — A floresta aleatória é, talvez, o modelo clássico **mais usado** em problemas operacionais de classificação a partir de **dados de sensores**. Para classificação de assinaturas — acústicas, radar, eletro-ópticas —, ela combina três virtudes raras de aparecerem juntas: **desempenho robusto sem ajuste fino**, **tolerância a atributos de escalas distintas** e, sobretudo, uma medida de **importância de atributos** que indica quais características do sinal mais pesaram na decisão. Essa última propriedade é *ouro para a auditabilidade*: permite ao analista verificar se o modelo se apoia em características **fisicamente plausíveis** do sinal — e não em algum artefato espúrio de aquisição.

### 2.3.3 Importância de atributos em código

A **Listagem 2.3** treina uma floresta aleatória sobre um problema de classificação de assinaturas e extrai a **importância dos atributos** — a contribuição relativa de cada característica do sinal para a decisão.

In [ ]:
# Listagem 2.3 - Floresta aleatoria e importancia de atributos.
# Classificacao de assinaturas e leitura da importancia dos atributos.
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

X, y = make_classification(
    n_samples=1500, n_features=12, n_informative=6, random_state=0
)
nomes = [f"banda_{j}" for j in range(X.shape[1])]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y
)

floresta = RandomForestClassifier(
    n_estimators=300, max_depth=None, random_state=0, n_jobs=-1
)
floresta.fit(X_tr, y_tr)
y_pred = floresta.predict(X_te)
print("F1:", round(f1_score(y_te, y_pred), 3))

# Importancia dos atributos, ordenada do mais ao menos influente
ordem = np.argsort(floresta.feature_importances_)[::-1]
print("\nImportancia dos atributos:")
for j in ordem[:6]:
    print(f"  {nomes[j]:>9}: {floresta.feature_importances_[j]:.3f}")

In [ ]:
# Visualizacao: importancia de atributos + arvore isolada vs. floresta
from sklearn.tree import DecisionTreeClassifier

fig, eixos = plt.subplots(1, 2, figsize=(12, 4))

# Grafico 1: importancia dos atributos (auditoria do modelo)
importancias = floresta.feature_importances_[ordem]
eixos[0].barh([nomes[j] for j in ordem][::-1], importancias[::-1])
eixos[0].set_xlabel("Importancia relativa")
eixos[0].set_title("Quais bandas do sinal mais pesam na decisao?")

# Grafico 2: sobreajuste da arvore isolada vs. robustez da floresta
arvore_unica = DecisionTreeClassifier(random_state=0).fit(X_tr, y_tr)
comparacao = {
    "Arvore unica":  (f1_score(y_tr, arvore_unica.predict(X_tr)),
                      f1_score(y_te, arvore_unica.predict(X_te))),
    "Floresta (300)": (f1_score(y_tr, floresta.predict(X_tr)),
                       f1_score(y_te, floresta.predict(X_te))),
}
rotulos = list(comparacao.keys())
f1_treino = [v[0] for v in comparacao.values()]
f1_teste  = [v[1] for v in comparacao.values()]
posicao = np.arange(len(rotulos))
eixos[1].bar(posicao - 0.18, f1_treino, width=0.36, label="F1 treino")
eixos[1].bar(posicao + 0.18, f1_teste,  width=0.36, label="F1 teste")
eixos[1].set_xticks(posicao, rotulos)
eixos[1].set_ylim(0.80, 1.02)
eixos[1].set_ylabel("F1-score")
eixos[1].set_title("Bagging reduz a lacuna treino-teste")
eixos[1].legend()

plt.tight_layout()
plt.show()

print("A arvore unica atinge F1 = 1.0 no treino (memorizou o ruido),")
print("mas cai no teste. A floresta generaliza melhor: o voto de centenas")
print("de arvores descorrelacionadas cancela os erros individuais.")

**Observe:** a floresta **não exige padronização** e tende a alcançar F1 elevado **sem qualquer ajuste de hiperparâmetros** — daí seu papel de *baseline forte* para classificação tabular.

A leitura da importância dos atributos, contudo, **exige cautela**: ela indica **associação** com a decisão, *não causalidade*, e pode ser inflada por atributos de alta cardinalidade. É um **ponto de partida para a investigação, não uma conclusão**.

---
## 2.4 Máquinas de vetores de suporte

A última família clássica — as **máquinas de vetores de suporte** (*support vector machines*, SVM) — parte de uma intuição **geométrica** distinta e particularmente elegante, e mantém relevância em um nicho importante: problemas com **muitos atributos e poucos exemplos**, comuns quando a rotulagem é cara, como na classificação de assinaturas raras.

### 2.4.1 Margem máxima

Em um problema binário linearmente separável há, em geral, **infinitos hiperplanos** que separam as classes. A SVM escolhe um critério específico: o hiperplano que **maximiza a margem** — a distância até os exemplos mais próximos de cada classe. Esses exemplos críticos, que tocam a margem, são os **vetores de suporte**, e dão nome ao método. A intuição: o separador de margem máxima é **o mais robusto**, pois é o que menos provavelmente erra em exemplos novos próximos da fronteira.

Formalmente, com rótulos $y_i \in \{-1, +1\}$, busca-se:

$$\min_{\mathbf{w}, b} \; \frac{1}{2} \lVert \mathbf{w} \rVert^2 \quad \text{sujeito a} \quad y_i \left( \mathbf{w}^\top \mathbf{x}_i + b \right) \geq 1, \; i = 1, \ldots, n$$

Minimizar $\lVert \mathbf{w} \rVert$ equivale a maximizar a margem. Como os dados raramente são perfeitamente separáveis, admite-se uma **margem suave** (*soft margin*), com variáveis de folga penalizadas por um hiperparâmetro $C$ que regula a troca entre *margem larga* e *erros no treino*.

In [ ]:
# Intuicao geometrica da SVM: margem maxima e vetores de suporte
from sklearn.datasets import make_blobs
from sklearn.svm import SVC

X_svm, y_svm = make_blobs(n_samples=80, centers=2, cluster_std=1.1,
                          random_state=SEMENTE)
svm_linear = SVC(kernel="linear", C=1000).fit(X_svm, y_svm)

plt.figure(figsize=(7.5, 6))
plt.scatter(X_svm[:, 0], X_svm[:, 1], c=y_svm, cmap="coolwarm",
            edgecolor="k", linewidth=0.4, s=45)

# Fronteira de decisao e margens: curvas de nivel da funcao de decisao
eixo = plt.gca()
xx = np.linspace(*eixo.get_xlim(), 60)
yy = np.linspace(*eixo.get_ylim(), 60)
XX, YY = np.meshgrid(xx, yy)
Z = svm_linear.decision_function(np.c_[XX.ravel(), YY.ravel()]).reshape(XX.shape)
eixo.contour(XX, YY, Z, levels=[-1, 0, 1], colors="k",
             linestyles=["--", "-", "--"], linewidths=[1, 2, 1])

# Vetores de suporte: os exemplos criticos que "seguram" a margem
eixo.scatter(svm_linear.support_vectors_[:, 0],
             svm_linear.support_vectors_[:, 1],
             s=220, facecolors="none", edgecolors="k", linewidths=1.6,
             label=f"vetores de suporte ({len(svm_linear.support_vectors_)})")

plt.title("SVM linear: fronteira (linha cheia), margens (tracejadas)\n"
          "e vetores de suporte (circulados)")
plt.legend(loc="best")
plt.tight_layout()
plt.show()

print("Apenas os vetores de suporte determinam a fronteira --")
print("os demais exemplos poderiam ser removidos sem alterar o modelo.")

### 2.4.2 O truque do kernel

A elegância da SVM revela-se quando os dados **não são linearmente separáveis** no espaço original. Em vez de construir explicitamente atributos não lineares, a SVM usa o **truque do kernel**: substitui o produto interno $\mathbf{x}_i^\top \mathbf{x}_j$ por uma **função de kernel** $K(\mathbf{x}_i, \mathbf{x}_j)$ que corresponde a um produto interno em um espaço de dimensão muito maior — *sem nunca calcular as coordenadas nesse espaço*. O kernel mais usado é o **RBF** (*radial basis function*, ou gaussiano):

$$K(\mathbf{x}_i, \mathbf{x}_j) = \exp\left( -\gamma \, \lVert \mathbf{x}_i - \mathbf{x}_j \rVert^2 \right)$$

Com ele, fronteiras de decisão **arbitrariamente complexas** tornam-se acessíveis — ao custo de **dois hiperparâmetros** a ajustar ($C$ e $\gamma$) e de uma sensibilidade real a esse ajuste. A célula abaixo compara, sobre o mesmo conjunto não linear, a SVM **linear** (que falha) e a SVM com kernel **RBF** (que resolve):

In [ ]:
# Kernel linear vs. kernel RBF em dados nao linearmente separaveis
from sklearn.datasets import make_moons

X_luas, y_luas = make_moons(n_samples=300, noise=0.25, random_state=SEMENTE)

fig, eixos = plt.subplots(1, 2, figsize=(12, 4.5))
for eixo, nucleo in zip(eixos, ["linear", "rbf"]):
    svm = SVC(kernel=nucleo, C=10.0, gamma="scale").fit(X_luas, y_luas)

    xx = np.linspace(X_luas[:, 0].min() - 0.5, X_luas[:, 0].max() + 0.5, 200)
    yy = np.linspace(X_luas[:, 1].min() - 0.5, X_luas[:, 1].max() + 0.5, 200)
    XX, YY = np.meshgrid(xx, yy)
    Z = svm.predict(np.c_[XX.ravel(), YY.ravel()]).reshape(XX.shape)

    eixo.contourf(XX, YY, Z, alpha=0.20, cmap="coolwarm")
    eixo.scatter(X_luas[:, 0], X_luas[:, 1], c=y_luas, cmap="coolwarm",
                 edgecolor="k", linewidth=0.3, s=28)
    acuracia = svm.score(X_luas, y_luas)
    eixo.set_title(f"kernel = '{nucleo}'  (acuracia no proprio conjunto: {acuracia:.2f})")

plt.suptitle("O truque do kernel: fronteiras nao lineares sem construir atributos novos")
plt.tight_layout()
plt.show()

A **Listagem 2.4** aplica a SVM com kernel RBF a um problema de maior dimensão, com um detalhe metodológico central: a **padronização acoplada ao modelo** via `Pipeline`.

> **✅ Boa prática** — Encapsular a padronização e o modelo em um único `Pipeline` do `scikit-learn` **não é mero estilo**: é a defesa mais eficaz contra a **fuga de informação** (*data leakage*) discutida no Capítulo 1. Sem o pipeline, é fácil padronizar o conjunto inteiro *antes* de separar treino e teste, deixando estatísticas do teste vazarem para o treino e produzindo **métricas otimistas que não se sustentam em produção**. Com o pipeline, a transformação é ajustada **apenas no treino** e reaplicada ao teste — automaticamente, a cada partição da validação cruzada.

In [ ]:
# Listagem 2.4 - SVM com kernel RBF e padronizacao acoplada.
# SVM com kernel RBF; padronizacao e' obrigatoria.
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report

X, y = make_classification(
    n_samples=1200, n_features=20, n_informative=8, random_state=0
)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y
)

# O pipeline garante que a padronizacao do teste use estatisticas do treino
modelo = make_pipeline(
    StandardScaler(),
    SVC(kernel="rbf", C=10.0, gamma="scale", class_weight="balanced")
)
modelo.fit(X_tr, y_tr)
print(classification_report(y_te, modelo.predict(X_te), digits=3))

### 2.4.3 Quando preferir cada família

Não há modelo universalmente melhor; **há adequação ao problema**. Como heurística inicial:

| Família | Primeira escolha quando... | Exige | Cuidados |
|---|---|---|---|
| **Regressão logística** | se deseja **interpretabilidade** e **probabilidades calibradas**, com fronteira aproximadamente linear | padronização | fronteiras não lineares ficam fora do alcance |
| **Floresta aleatória** | dados **tabulares heterogêneos de sensores** — baseline robusta | quase nada (sem padronização) | importância de atributos ≠ causalidade |
| **SVM com kernel** | **alta dimensionalidade com poucos exemplos** (rotulagem cara) | padronização + ajuste de $C$ e $\gamma$ | não escala bem para conjuntos muito grandes |

A decisão final raramente se toma no papel: **toma-se comparando os modelos empiricamente**, como faremos a seguir.

---
## 2.5 Aplicação integrada: classificação de ameaças a partir de sensores

Encerramos o capítulo reunindo as famílias em um único **pipeline comparativo** — o gesto metodológico central de todo projeto operacional. Simulamos um problema de **classificação de ameaças a partir de dados de sensores**: cada exemplo reúne características extraídas de um contato (parâmetros cinemáticos, atributos espectrais, índices de qualidade do sinal), e a tarefa é classificá-lo como *ameaça ou não*, sob **desbalanceamento realista** (15% de positivos).

A **Listagem 2.5** treina e avalia, sob **validação cruzada estratificada**, os modelos do capítulo, ordenando-os pelo **F1-score da classe de interesse** — a métrica que melhor equilibra precisão e revocação na detecção da ameaça.

In [ ]:
# Listagem 2.5 - Comparacao sistematica de modelos classicos sob validacao cruzada.
# Comparacao de modelos classicos sob validacao cruzada estratificada.
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Problema sintetico: deteccao de ameaca, 15% de exemplos positivos
X, y = make_classification(
    n_samples=2500, n_features=16, n_informative=8,
    weights=[0.85, 0.15], random_state=0
)

modelos = {
    "Regressao logistica": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, class_weight="balanced")),
    "Floresta aleatoria": RandomForestClassifier(
        n_estimators=300, class_weight="balanced",
        random_state=0, n_jobs=-1),
    "SVM (RBF)": make_pipeline(
        StandardScaler(),
        SVC(kernel="rbf", C=10.0, gamma="scale", class_weight="balanced")),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
resultados = {}
for nome, modelo in modelos.items():
    f1 = cross_val_score(modelo, X, y, cv=cv, scoring="f1")
    resultados[nome] = (f1.mean(), f1.std())

print("Modelo                  F1 medio   desvio")
for nome, (media, dp) in sorted(
        resultados.items(), key=lambda kv: kv[1][0], reverse=True):
    print(f"  {nome:<22} {media:6.3f}   +/- {dp:.3f}")

In [ ]:
# Visualizacao da comparacao: F1 medio com a variabilidade entre particoes
nomes_ord = sorted(resultados, key=lambda k: resultados[k][0])
medias = [resultados[k][0] for k in nomes_ord]
desvios = [resultados[k][1] for k in nomes_ord]

plt.figure(figsize=(8, 3.5))
plt.barh(nomes_ord, medias, xerr=desvios, capsize=5, height=0.5)
plt.xlabel("F1-score da classe 'ameaca' (validacao cruzada, 5 particoes)")
plt.title("Comparacao honesta: media E variabilidade, nao um numero unico")
plt.xlim(0, 1)
for i, (m, d) in enumerate(zip(medias, desvios)):
    plt.text(m + d + 0.02, i, f"{m:.3f}", va="center")
plt.tight_layout()
plt.show()

print("A escolha final pesa nao so o F1 medio, mas tambem a variancia")
print("(estabilidade entre particoes), o custo de inferencia e --")
print("sempre, em defesa -- a auditabilidade.")

**Observe:** o resultado típico confirma a lição central do capítulo — a **floresta aleatória** costuma liderar ou empatar em problemas tabulares heterogêneos, com pouco ou nenhum ajuste; a **SVM com kernel** se aproxima quando há estrutura não linear forte; e a **regressão logística**, embora muitas vezes inferior em F1, permanece valiosa por sua **interpretabilidade** e por suas **probabilidades**.

> **🛡️ Contexto de defesa** — Repare no que o pipeline comparativo **não** faz: ele *não elege um vencedor de forma automática*. Em um sistema crítico, a métrica é **apenas uma das entradas** da decisão. Um modelo 1% superior em F1, mas inauditável, que exige hardware dedicado de inferência e cujo comportamento se degrada sob *drift* de sensor, pode ser a **pior** escolha operacional, ainda que a melhor no notebook. **A engenharia de defesa começa onde a competição de métricas termina.**

---
## 2.6 O caminho à frente

No **Capítulo 3**, fecharemos o Módulo I com o salto para o *deep learning*: do perceptron ao *multilayer perceptron*, do algoritmo de retropropagação às primeiras arquiteturas profundas, com implementação em TensorFlow e PyTorch. As redes neurais **não tornam obsoletos** os modelos deste capítulo — elas ampliam o repertório, e a disciplina de comparar contra uma **baseline clássica forte** permanece tão necessária quanto antes.

---

## 📋 Resumo do capítulo

- A **regressão linear** modela uma saída contínua como combinação linear dos atributos. Sua transparência — coeficientes com interpretação direta — faz dela a **baseline natural** de todo problema de regressão, e instrumento de auditoria em manutenção preditiva.

- A **regressão logística** adapta a combinação linear à classificação, devolvendo **probabilidades** via função sigmoide. A probabilidade é *informação operacional*, e o **limiar de decisão é uma alavanca a calibrar** — não um valor fixo.

- Em **classes desbalanceadas** — a regra, e não a exceção, em detecção de ameaças —, a **acurácia agregada engana**. Avalie por precisão, revocação, F1 e matriz de confusão, ponderando o custo operacional distinto de falsos positivos e falsos negativos.

- **Árvores de decisão** capturam relações não lineares e são legíveis, mas **sobreajustam**. A **floresta aleatória** corrige isso por *bagging* e amostragem de atributos, oferecendo desempenho robusto e **importância de atributos** — valiosa para auditoria.

- As **máquinas de vetores de suporte** buscam o separador de **margem máxima** e, via **truque do kernel** (RBF), acessam fronteiras não lineares. Brilham em alta dimensão com poucos exemplos, mas exigem padronização e ajuste de $C$ e $\gamma$.

- **Não há modelo universalmente superior.** A prática correta é comparar modelos empiricamente, sob **validação cruzada estratificada**, e decidir pesando F1, variância, custo de inferência e **auditabilidade** — pois, em defesa, a melhor escolha de notebook nem sempre é a melhor escolha operacional.

## ⚠️ Armadilhas comuns

1. **Avaliar problema desbalanceado por acurácia.** Com classe rara, responder sempre "classe majoritária" produz acurácia alta e utilidade nula. Use métricas por classe e analise a matriz de confusão à luz do custo operacional de cada tipo de erro.
2. **Esquecer de padronizar os atributos.** Regressão logística e SVM são sensíveis à escala. Encapsule a padronização no pipeline para que seja sempre aplicada de forma correta.
3. **Padronizar antes de separar treino e teste.** Ajustar o padronizador sobre o conjunto inteiro deixa estatísticas do teste vazarem para o treino (*data leakage*). Ajuste sempre no treino e reaplique ao teste — o que o `Pipeline` faz automaticamente.
4. **Confiar cegamente na importância de atributos.** A importância da floresta indica **associação, não causalidade**, e pode ser inflada por atributos de alta cardinalidade. É ponto de partida de investigação, não veredito.
5. **Pular a baseline.** Apresentar um modelo complexo sem compará-lo a uma referência simples impede saber se ele de fato ajuda — e esconde erros de pipeline que uma baseline revelaria de imediato.
6. **Tratar o limiar de 0,5 como sagrado.** O limiar de decisão é ajustável e deve refletir o **custo relativo dos erros**. Em ameaças raras de alto custo, frequentemente se baixa o limiar para aumentar a revocação, aceitando mais falsos positivos.

---
## 🧭 Exercícios

Classificação: **Essencial** (fixação) · **Tático** (aplicação) · **Estratégico** (extensão criativa). Os exercícios de código têm células preparadas abaixo; os dissertativos podem ser respondidos em células de texto no próprio notebook.

### Essencial

**Exercício 2.1** — Execute a célula abaixo (cópia da Listagem 2.1). Compare os coeficientes estimados com os valores que geraram os dados (`0.18`, `7.5`, `0.9`). Em seguida, **aumente o ruído** de `12` para `40` e relate, em duas linhas, o efeito sobre o $R^2$ e sobre a proximidade dos coeficientes.

In [ ]:
# Exercicio 2.1 - aumente o ruido e observe o efeito sobre R2 e coeficientes
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

rng = np.random.default_rng(0)
n = 400
distancia  = rng.uniform(50, 800, n)
carga      = rng.uniform(0.5, 8.0, n)
velocidade = rng.uniform(40, 90, n)

consumo = (0.18 * distancia + 7.5 * carga + 0.9 * velocidade
           + rng.normal(0, 12, n))   # <--- ALTERE o desvio de 12 para 40

X = np.column_stack([distancia, carga, velocidade])
X_tr, X_te, y_tr, y_te = train_test_split(X, consumo, test_size=0.25,
                                          random_state=0)
modelo = LinearRegression().fit(X_tr, y_tr)

print("Coeficientes verdadeiros: [0.18  7.5   0.9]")
print("Coeficientes estimados:  ", np.round(modelo.coef_, 3))
print("R2:", round(r2_score(y_te, modelo.predict(X_te)), 3))

# Sua resposta (2 linhas):
# 1) Efeito sobre o R2: ...
# 2) Efeito sobre os coeficientes: ...

**Exercício 2.2** — Na célula abaixo (cópia da Listagem 2.2), **remova** o parâmetro `class_weight="balanced"` e reexecute. Compare a **revocação da classe hostil** antes e depois. Explique, em uma frase, por que o modelo sem ponderação tende a "ignorar" a classe rara.

In [ ]:
# Exercicio 2.2 - remova class_weight="balanced" e compare a revocacao
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

X, y = make_classification(
    n_samples=2000, n_features=10, n_informative=5,
    weights=[0.88, 0.12], random_state=0
)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y
)
escalador = StandardScaler().fit(X_tr)
X_tr_n, X_te_n = escalador.transform(X_tr), escalador.transform(X_te)

modelo = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"   # <--- REMOVA este parametro e reexecute
)
modelo.fit(X_tr_n, y_tr)
print(classification_report(y_te, modelo.predict(X_te_n), digits=3,
      target_names=["neutro", "hostil"]))

# Sua resposta (1 frase):
# O modelo sem ponderacao tende a ignorar a classe rara porque ...

**Exercício 2.3** — Execute a célula abaixo e liste os **três atributos mais importantes**. Em seguida, compare o F1 da **árvore de decisão única** ao da **floresta**. Em duas linhas, relacione a diferença observada ao conceito de *bagging*.

In [ ]:
# Exercicio 2.3 - arvore unica vs. floresta: o efeito do bagging
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score

X, y = make_classification(
    n_samples=1500, n_features=12, n_informative=6, random_state=0
)
nomes = [f"banda_{j}" for j in range(X.shape[1])]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y
)

floresta = RandomForestClassifier(n_estimators=300, random_state=0,
                                  n_jobs=-1).fit(X_tr, y_tr)
arvore = DecisionTreeClassifier(random_state=0).fit(X_tr, y_tr)

print("F1 floresta:    ", round(f1_score(y_te, floresta.predict(X_te)), 3))
print("F1 arvore unica:", round(f1_score(y_te, arvore.predict(X_te)), 3))

ordem = np.argsort(floresta.feature_importances_)[::-1]
print("\n3 atributos mais importantes:")
for j in ordem[:3]:
    print(f"  {nomes[j]}: {floresta.feature_importances_[j]:.3f}")

# Sua resposta (2 linhas):
# A diferenca de F1 se relaciona ao bagging porque ...

### Tático

**Exercício 2.4** — A célula abaixo varia o hiperparâmetro $C$ da SVM entre $\{0{,}1;\ 1;\ 10;\ 100\}$ e registra o F1 da classe positiva em cada caso. Execute-a e descreva, em poucas linhas, o comportamento observado, relacionando-o à **troca entre margem larga e erros no treino** regulada por $C$.

In [ ]:
# Exercicio 2.4 - busca manual do hiperparametro C da SVM
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import f1_score

X, y = make_classification(
    n_samples=1200, n_features=20, n_informative=8, random_state=0
)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y
)

print("   C    | F1 (classe positiva)")
for C in [0.1, 1, 10, 100]:
    modelo = make_pipeline(
        StandardScaler(),
        SVC(kernel="rbf", C=C, gamma="scale", class_weight="balanced")
    ).fit(X_tr, y_tr)
    f1 = f1_score(y_te, modelo.predict(X_te))
    print(f"  {C:5.1f} |  {f1:.3f}")

# Sua resposta (poucas linhas):
# C pequeno -> margem larga, mais erros tolerados no treino: ...
# C grande  -> margem estreita, poucos erros tolerados: ...

**Exercício 2.5** — Estenda a comparação da Listagem 2.5 com a **curva precision–recall** do melhor modelo. A célula abaixo já obtém as probabilidades sob validação cruzada e traça a curva; **identifique um limiar que privilegie a revocação da ameaça acima de 0,90** e relate o preço pago em precisão.

In [ ]:
# Exercicio 2.5 - curva precision-recall e escolha de limiar operacional
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_recall_curve
import matplotlib.pyplot as plt

X, y = make_classification(
    n_samples=2500, n_features=16, n_informative=8,
    weights=[0.85, 0.15], random_state=0
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
melhor = RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                random_state=0, n_jobs=-1)

# Probabilidades previstas fora da amostra (cada exemplo previsto
# pela particao em que ele NAO participou do treino)
probas = cross_val_predict(melhor, X, y, cv=cv,
                           method="predict_proba")[:, 1]
precisao, revocacao, limiares = precision_recall_curve(y, probas)

plt.figure(figsize=(7, 4.5))
plt.plot(revocacao, precisao, lw=2)
plt.axvline(0.90, color="r", ls="--", lw=1.2,
            label="revocacao alvo >= 0.90")
plt.xlabel("Revocacao da ameaca (recall)")
plt.ylabel("Precisao")
plt.title("Curva precision-recall do melhor modelo")
plt.legend()
plt.tight_layout()
plt.show()

# Menor limiar cuja revocacao ainda e >= 0.90 e o preco pago em precisao
mascara = revocacao[:-1] >= 0.90
indice = np.where(mascara)[0][-1]
print(f"Limiar sugerido:      {limiares[indice]:.3f}")
print(f"Revocacao nesse ponto: {revocacao[indice]:.3f}")
print(f"Precisao nesse ponto:  {precisao[indice]:.3f}")

# Sua resposta (poucas linhas):
# Para garantir revocacao >= 0.90, o limiar cai para ... e a precisao
# paga o preco de ..., o que significa, operacionalmente, que ...

**Exercício 2.6** *(dissertativo)* — Considere o problema operacional de **previsão de falha de componente** a partir de histórico de operação (variáveis contínuas de temperatura, vibração e horas de uso). Decida se você o atacaria como **regressão** (tempo até a falha) ou **classificação** (falha nas próximas N horas), justifique a escolha e indique, para cada caso, a **métrica de avaliação adequada** e o motivo.

*Responda em uma célula de texto abaixo.*

### Estratégico

**Exercício 2.7** — Em um breve texto técnico (até duas páginas), discuta a **tensão entre desempenho e auditabilidade** na escolha de um modelo para um sistema de apoio à triagem de contatos. Construa um argumento, fundamentado nos conceitos do capítulo, sobre em que circunstâncias um modelo menos preciso, mas interpretável, deve ser preferido a um modelo mais preciso e opaco — e proponha **critérios objetivos** para essa decisão.

**Exercício 2.8** — Projete (em pseudocódigo ou diagrama) um **pipeline de validação que respeite a estrutura temporal** de dados de sensores — isto é, que evite usar informação futura para prever o passado. Explique por que a validação cruzada estratificada padrão da Listagem 2.5 seria **inadequada para uma série temporal**, e que estratégia (por exemplo, validação por janelas deslizantes) a substituiria.

**Exercício 2.9** — Escolha um conjunto de dados tabular de domínio público (de sensores, logística ou meteorologia) e conduza um **estudo comparativo completo** dos modelos do capítulo, redigindo um miniartigo de até três páginas que documente: a definição do problema, a estratégia de avaliação, os resultados com sua variância, e uma recomendação fundamentada — incluindo a discussão explícita dos **vieses do conjunto** e das **limitações de generalização** para um contexto operacional real.

---

*Fim do Capítulo 2 — no Capítulo 3, o salto para as redes neurais e o deep learning.*